# 框架

```mermaid
graph TD
    %% ------------------- 样式定义 (Style Definitions) -------------------
    classDef interface fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
    classDef foundation fill:#D5F5E3,stroke:#287C3E,stroke-width:2px
    classDef writer fill:#FAD7A0,stroke:#333,stroke-width:2px
    classDef reader fill:#E8DAEF,stroke:#333,stroke-width:2px
    classDef build fill:#EEEEEE,stroke:#888,stroke-width:1px

    %% ------------------- 顶层容器 (Top Level Subgraph) -------------------
    subgraph WtDataStorageAD ["高性能数据存储模块 (WtDataStorageAD) - 基于 LMDB 和内存映射"]
        direction LR

        %% ------------------- 1. 外部接口 (External Interfaces) -------------------
        subgraph ExternalInterfaces ["外部接口 (定义于 Includes/)"]
            direction TB
            IDataWriter("IDataWriter.h<br/><b>[接口]</b> 实时数据写入器<br/><b>意义:</b> 定义数据写入规范, 是数据生产者的统一入口"):::interface
            IDataReader("IDataReader.h<br/><b>[接口]</b> 实时数据读取器<br/><b>意义:</b> 定义实盘策略读取数据的规范 (实时+历史)"):::interface
            IBtDtReader("IBtDtReader.h<br/><b>[接口]</b> 回测数据读取器<br/><b>意义:</b> 定义回测引擎读取数据的规范 (性能优先)"):::interface
            IRdmDtReader("IRdmDtReader.h<br/><b>[接口]</b> 随机数据读取器<br/><b>意义:</b> 定义分析工具按需读取数据的规范 (灵活性优先)"):::interface
        end

        %% ------------------- 2. 模块内部基础 (Internal Foundation) -------------------
        subgraph InternalFoundation ["模块内部基础 (Internal Foundation)"]
            direction TB
            DataDefine("DataDefineAD.h<br/><b>[数据结构]</b> 内存缓存定义<br/><b>作用:</b> 定义内存映射的实时缓存结构<br/>(RTTickCache, RTBarCache)<br/><b>用法:</b> WtDataWriterAD 写入, WtDataReaderAD 读取"):::foundation
            LMDBKeys("LMDBKeys.h<br/><b>[数据结构]</b> LMDB键定义<br/><b>作用:</b> 定义数据库键结构 (LMDBHftKey, LMDBBarKey)<br/>包含字节序转换 (reverseEndian), 确保时间排序正确"):::foundation
        end

        %% ------------------- 3. 核心组件 (Core Components) -------------------
        subgraph CoreComponents ["核心实现 (Implementations)"]
            direction TB

            %% 3.1 写入器 (Writer)
            subgraph Writer ["数据写入 (Producer)<br/>职责: 接收实时数据, 写入缓存和数据库, 合成K线"]
                direction TB
                Writer_h("WtDataWriterAD.h<br/><b>[定义]</b> 高级数据写入器 (AD)"):::writer
                Writer_cpp("WtDataWriterAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 接收实时Tick, 异步写入LMDB, 合成K线, 更新实时缓存"):::writer
            end

            %% 3.2 实时读取器 (Real-time Reader)
            subgraph RlReader ["实时读取 (Real-time Consumer)<br/>职责: 供实盘策略使用"]
                direction TB
                Reader_h("WtDataReaderAD.h<br/><b>[定义]</b> 实时数据读取器 (AD)"):::reader
                Reader_cpp("WtDataReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 融合LMDB(历史)+内存缓存(当日)+内存(最新)三级数据"):::reader
            end
            
            %% 3.3 回测读取器 (Backtest Reader)
            subgraph BtReader ["回测读取 (Backtest Consumer)<br/>职责: 供回测引擎使用"]
                direction TB
                BtReader_h("WtBtDtReaderAD.h<br/><b>[定义]</b> 回测数据读取器 (AD)"):::reader
                BtReader_cpp("WtBtDtReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 高速读取原始(raw)二进制数据块, 极致回测性能"):::reader
            end

            %% 3.4 随机读取器 (Random-access Reader)
            subgraph RdmReader ["随机读取 (Analysis Consumer)<br/>职责: 供数据分析工具使用"]
                direction TB
                RdmReader_h("WtRdmDtReaderAD.h<br/><b>[定义]</b> 随机数据读取器 (AD)"):::reader
                RdmReader_cpp("WtRdmDtReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 按需(ByRange, ByCount)加载数据到内存"):::reader
            end
        end
    end

    %% ------------------- 4. 关系连接 (Relationships) -------------------

    %% 4.1 接口实现关系 (Implementation)
    IDataWriter -- "实现 (implements)" --> Writer_h
    IDataReader -- "实现 (implements)" --> Reader_h
    IBtDtReader -- "实现 (implements)" --> BtReader_h
    IRdmDtReader -- "实现 (implements)" --> RdmReader_h

    %% 4.2 头文件与实现文件 (h/cpp)
    Writer_h --> Writer_cpp
    Reader_h --> Reader_cpp
    BtReader_h --> BtReader_cpp
    RdmReader_h --> RdmReader_cpp

    %% 4.3 核心依赖关系 (Core Dependencies)
    Writer_h -- "依赖 (uses)" --> DataDefine
    Writer_h -- "依赖 (uses)" --> LMDBKeys
    
    Reader_h -- "依赖 (uses)" --> DataDefine
    Reader_h -- "依赖 (uses)" --> LMDBKeys
    
    BtReader_h -- "依赖 (uses)" --> LMDBKeys
    RdmReader_h -- "依赖 (uses)" --> LMDBKeys
    
    %% 4.4 逻辑数据流 (Logical Data Flow)
    Writer_cpp -- "写入 (Writes to)" --> DataDefine
    Writer_cpp -- "写入 (Writes to)" --> LMDBKeys
    Reader_cpp -- "读取 (Reads from)" --> DataDefine
    Reader_cpp -- "读取 (Reads from)" --> LMDBKeys
    BtReader_cpp -- "读取 (Reads from)" --> LMDBKeys
    RdmReader_cpp -- "读取 (Reads from)" --> LMDBKeys
```

# 存储格式定义 DataDefineAD.h

- **核心常量与版本控制**
  - `const char BLK_FLAG[] = "&^%$#@!\0"`
    - **作用**：8 字节的“魔数”（Magic Number），用作文件的指纹。
    - **意义**：当 `WtDataReaderAD` 加载一个缓存文件时，它会首先检查文件头部是否有这个 `BLK_FLAG`。如果匹配，则确认这是一个有效的数据缓存文件；如果不匹配，则认为文件已损坏或格式不正确，拒绝加载。
  - `#define BLOCK_VERSION_RAW 1`
    - **作用**：定义了缓存文件的数据格式版本号。
    - **意义**：`RAW` 表示文件中的数据（如 `WTSTickStruct`, `WTSBarStruct`）是以**原始二进制 (raw)** 格式存储的，没有经过压缩。这为未来的格式升级（如添加压缩版本）提供了兼容性基础。

- **数据块类型枚举 (`BlockType`)**
  ```cpp
  typedef enum tagBlockType
  {
    BT_RT_Cache = 4
  } BlockType;
  ```
  - **作用**：标识数据块中存储的是哪一种数据。
  - **意义**：这是此文件中唯一定义的类型，代表**“实时缓存” (Real-Time Cache)**。
  - **区别**：这与标准版 `DataDefine.h` 中的 `BT_HIS_Minute1` (历史K线)、`BT_HIS_Ticks` (历史Tick) 等类型是**完全不同**的。`WtDataStorageAD` 模块使用 LMDB 存储历史数据，因此它不需要历史数据块类型，它只需要这个实时缓存类型来管理当日的内存映射文件。

- **数据块头部结构**
  ```cpp
  typedef struct _BlockHeader
  {
    char		_blk_flag[FLAG_SIZE]; // 数据块标识魔数，用于验证数据块的有效性
    uint16_t	_type;  // 数据块类型，对应BlockType枚举值
    uint16_t	_version; // 数据块版本号，用于格式兼容性管理
  } BlockHeader;
  ```
  - **作用**：作为所有实时缓存文件的通用元信息，提供了识别和解析数据的基础。
  - **意义**：这是所有缓存文件的基础身份信息。
  ```cpp
  typedef struct _RTBlockHeader : BlockHeader
  {
    uint32_t _size; // 当前已使用的数据项数量
    uint32_t _capacity; // 数据块的最大容量（数据项数量）
  } RTBlockHeader;
  ```
  - **定义**：继承自 `BlockHeader`，并增加了 `uint32_t _size` (当前条数) 和 `uint32_t _capacity` (总容量) 字段。
  - **区别与意义**：这是 `WtDataStorageAD` 缓存设计的核心。
    - `_capacity`：在文件创建时，会预先分配 `_capacity` * `sizeof(Item)` 的巨大空间。
    - `_size`：在交易过程中，`WtDataWriterAD` 每写入一条新数据，只会将 `_size` 加 1，而**不需要扩展文件**。
    - 这种“预分配容量、递增大小”的机制，避免了在行情高峰期频繁进行昂贵的文件 I/O 和扩容操作，是实现高性能写入的关键。

- **具体数据缓存项 (`CacheItem`) 定义**
  ```cpp
  typedef struct _TickCacheItem
  {
    uint32_t		_date;  // 交易日期（格式：YYYYMMDD）
    WTSTickStruct	_tick;  // Tick行情数据结构
  } TickCacheItem;
  ```
  - **意义**：这是实时 Tick 缓存 (`cache_tick.dmb`) 中每一条记录的格式。它将 `WTSTickStruct`（包含完整的行情快照）与一个交易日期 `_date` 绑定在一起。
  ```cpp
  typedef struct _BarCacheItem
  {
    char			_exchg[16]; // 交易所代码，固定16字节长度
    char			_code[32];  // 合约代码，固定32字节长度
    WTSBarStruct	_bar; // K线数据结构
  } BarCacheItem;
  ```
  - **意义**：这是实时 K 线缓存 (`cache_m1.dmb`, `cache_m5.dmb` 等) 中每一条记录的格式。

- **实时缓存块实现 (Flexible Array Member)**
  - **作用**：将头部与数据项结合起来，定义了完整的内存映射文件布局。
  - **用法**：使用了 C 语言中的“柔性数组成员”技巧 (`_items[0]`)。这意味着头部和它后面的 `_items` 数组存储在一块**连续的内存**中。
  ```cpp
  typedef struct _RTTickCache : RTBlockHeader
  {
    TickCacheItem	_items[0];  // 变长数组，存储Tick缓存项数据
  } RTTickCache;
  ```
  - **定义**：`RTBlockHeader` 后面紧跟着 `TickCacheItem _items[0]`。
  - **意义**：定义了 `cache_tick.dmb` 文件的内存布局。
  ```cpp
  typedef struct _RTBarCache : RTBlockHeader
  {
    BarCacheItem	_items[0];  // 变长数组，存储K线缓存项数据
  } RTBarCache;
  ```
  - **定义**：`RTBlockHeader` 后面紧跟着 `BarCacheItem _items[0]`。
  - **意义**：定义了 `cache_m1.dmb`, `cache_m5.dmb`, `cache_d1.dmb` 等 K 线缓存文件的内存布局。

# LMDB数据库键定义 LMDBKeys.h
**LMBD数据库** 参考 [WTSUtils/note.ipynb/键值型数据库 lmdb](../WTSUtils/note.ipynb)

**WtDataStorageAD模块** 使用 LMDB 数据库来存储数据
- LMDB 是一个高性能的 *键值(Key-Value)数据库*
  - Value：要存储的数据，即 WTSStruct.h 中定义的 `WTSTickStruct` (Tick数据) 和 `WTSBarStruct` (K线数据)。
  - Key：用来查找数据的索引

**LMDBKeys.h** 的作用就是定义索引 (Key) 的二进制结构
- **小端序**：对于一个字节块，例如 uint32_t，低位字节在存储空间中排在前面。**大端序**相反
  - 例如：整数 20230601（0x013553C1），小端序存储为 C1 53 35 01
- LMDB 会自动对所有的 Key 进行排序：默认方式是 *字典序*，即逐个字节进行比较
  - 如果系统是小端序，例如 20230601（0x013553C1，存储为 1F 50 35 01）和 20230531（0x0135501F，存储为 C1 53 35 01）
  - 会判断为 20230601 小于 20230531，这是不符合逻辑的

## 字节序转换
```cpp
/**
 * @brief 16位整数字节序转换函数
 * 
 * 将16位整数从小端字节序转换为大端字节序，用于确保LMDB键值的
 * 字典序与数值大小顺序一致。这对于时间相关的查询优化至关重要。
 * 
 * @param src 源16位整数（小端字节序）
 * @return 转换后的16位整数（大端字节序）
 */
static uint16_t reverseEndian(uint16_t src)
{
	uint16_t up = (src & 0x00FF) << 8;         // 低字节移到高位
	uint16_t low = (src & 0xFF00) >> 8;        // 高字节移到低位
	return up + low;                           // 合并结果
}

/**
 * @brief 32位整数字节序转换函数
 * 
 * 将32位整数从小端字节序转换为大端字节序，主要用于日期和时间
 * 字段的转换，确保时间序列数据在LMDB中的正确排序。
 * 
 * @param src 源32位整数（小端字节序）
 * @return 转换后的32位整数（大端字节序）
 */
static uint32_t reverseEndian(uint32_t src)
{
	uint32_t x = (src & 0x000000FF) << 24;     // 字节0移到位置3
	uint32_t y = (src & 0x0000FF00) << 8;      // 字节1移到位置2
	uint32_t z = (src & 0x00FF0000) >> 8;      // 字节2移到位置1
	uint32_t w = (src & 0xFF000000) >> 24;     // 字节3移到位置0
	return x + y + z + w;                      // 合并所有字节
}
```

## LMDB高频Tick数据的键
```cpp
typedef struct _LMDBHftKey
{
	char		_exchg[MAX_EXCHANGE_LENGTH];    // 交易所代码，如"SHFE"、"DCE"等
	char		_code[MAX_INSTRUMENT_LENGTH];   // 合约代码，如"rb2305"、"IF2303"等
	uint32_t	_date;                          // 交易日期，格式YYYYMMDD（大端序）
	uint32_t	_time;                          // 交易时间，格式HHMMSSsss（大端序）
} LMDBHftKey;
```
- 用作 Tick 数据、逐笔委托、逐笔成交等高频数据 (HFT) 的键。
- 键的排序优先级：`_exchg` \> `_code` \> `_date` \> `_time`。

## LMDB的K线的键
```cpp
typedef struct  _LMDBBarKey
{
public:
	char		_exchg[MAX_EXCHANGE_LENGTH];    // 交易所代码，如"SHFE"、"DCE"等
	char		_code[MAX_INSTRUMENT_LENGTH];   // 合约代码，如"rb2305"、"IF2303"等
	uint32_t	_bartime;                       // K线时间戳（大端序）
} LMDBBarKey;
```
- 用作 K 线 (`WTSBarStruct`) 数据
- 键的排序优先级：`_exchg` \> `_code` \> `_bartime`。

# 数据读取器 WtDataReaderAD.h/cpp
```cpp
class WtDataReader : public IDataReader
```
参考 [Includes/note.ipynb/数据管理接口层/数据读取 IDataReader.h/数据读取接口类 IDataReader](../Includes/note.ipynb)

`WtDataReaderAD` 是一个**三级缓存**的实时数据读取器，专为**实盘策略** (`IDataReader` 接口) 设计。
- 核心作用是融合**历史数据 (L3)** 和**实时数据 (L2)**
  - L2 数据（共享内存）会转储到 L3（固定文件存储）
  - L1 数据由 L2 和 L3 更新（环形缓冲区，方便访问）
- 并为策略提供一个极高性能的**内存缓存 (L1)**，以响应策略的 `get_tick_slice` 和 `get_kline_slice` 请求。

## 成员
- **核心管理器指针**
  - `std::string _base_dir`
    - **作用**：存储数据存储的根目录路径（例如 `"./data/"`）。
    - **意义**：用于定位 L2 实时缓存文件（如 `cache_m1.dmb`）和 L3 LMDB 数据库的物理位置。
  - `IBaseDataMgr* _base_data_mgr`
    - **作用**：指向基础数据管理器。
    - **意义**：用于查询合约静态信息（如 `WTSContractInfo`）和交易时段（`WTSSessionInfo`）。
  - `IHotMgr* _hot_mgr`
    - **作用**：指向主力合约管理器。
    - **意义**：用于解析连续合约代码（如将 "SHFE.rb.HOT" 解析为 "SHFE.rb2410"）。

- **L2 实时 K 线缓存 (内存映射文件)**
  - `RTBarCacheWrapper _m1_cache`：实时 1 分钟 K 线缓存。
  - `RTBarCacheWrapper _m5_cache`：实时 5 分钟 K 线缓存。
  - `RTBarCacheWrapper _d1_cache`：实时日 K 线缓存。
    - **作用**：这三个成员代表了 `WtDataWriterAD` 模块 正在实时写入的内存映射文件（`.dmb` 文件）
    ```cpp
    typedef struct _RTBarCacheWrapper
    {
      StdUniqueMutex _mtx;  // 互斥锁，保证线程安全访问
      std::string _filename;  // 缓存文件名
      wt_hashmap<std::string, uint32_t> _idx; // 合约索引映射表（key: "交易所.合约"）
      BoostMFPtr _file_ptr; // 内存映射文件指针
      RTBarCache* _cache_block; // 缓存数据块指针
      uint32_t _last_size;  // 上次检查时的缓存大小
    } RTBarCacheWrapper;
    ```

- **L1 内存缓存 (策略直接访问)**
  - `BarsCache _bars_cache`：K 线数据内存缓存（L1）。
    - typedef wt_hashmap\<std::string, `BarsList`\> BarsCache
    ```cpp
    typedef struct _BarsList
    {
      std::string		_exchg; // 交易所代码
      std::string		_code;  // 合约代码
      WTSKlinePeriod	_period;  // K线周期
      bool			_last_from_cache; // 最后一条K线是否来自实时缓存
      uint64_t		_last_req_time; // 最后请求时间戳

      boost::circular_buffer<WTSBarStruct>	_bars;	///< K线数据循环缓冲区
    } BarsList;
    ```
    - **融合了 L3 (LMDB的历史K线) 和 L2 (实时缓存的最新K线)，策略读取时无需区分历史和实时，实现了统一访问**。

  - `TicksCache _ticks_cache`：Tick 数据内存缓存（L1）
    - typedef wt_hashmap\<std::string, `TicksList`\> TicksCache
    ```cpp
    typedef struct _TicksList
    {
      std::string		_exchg; // 交易所代码
      std::string		_code;  // 合约代码
      uint64_t		_last_req_time; // 最后请求时间戳
      boost::circular_buffer<WTSTickStruct>	_ticks; // Tick数据循环缓冲区
    } TicksList;
    ```
    - 与 `BarsCache` 类似，它融合了 L3 的历史 Tick 和 L2 的实时 Tick

- **状态管理**
  - `uint64_t _last_time`
    - **作用**：记录 `onMinuteEnd` 事件最后一次处理的时间戳。
    - **意义**：用于事件去重，确保 `onMinuteEnd` 在同一分钟内只被执行一次。

- **L3 存储 (LMDB 数据库句柄)**
  - `WtLMDBMap _exchg_m1_dbs`：1 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_m5_dbs`：5 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_d1_dbs`：日 K 线数据库连接池。
  - `WtLMDBMap _tick_dbs`：Tick 数据库连接池。
    - typedef wt_hashmap\<std::string, `WtLMDBPtr`\> WtLMDBMap：数据库映射表类型
    - typedef std::shared_ptr\<`WtLMDB`\> WtLMDBPtr：LMDB数据库智能指针类型
  - 这些成员用于缓存 LMDB 数据库的连接句柄 (`WtLMDBPtr`)。
    - 当第一次请求某个交易所（K线）或合约（Tick）的数据时，会打开对应的 LMDB 数据库并缓存句柄；
    - 后续请求将直接重用此句柄，避免了频繁打开/关闭数据库文件的 I/O 开销。

## 方法

### IDataReader 接口实现

#### 初始化读取器 init

#### 从 L1-Tick缓存中读取Tick数据 readTickSlice

流程：
- 获取Tick的L1缓存 `_ticks_cache` 中对应数据块的引用 tickList
- 检查是否发生重载：
  - 全量：请求的 count 数大于 tickList 的当前容量（清空该L1缓存、扩容到count）
  - 增量：count数小于等于该L1缓存当前容量，但是最后储存时间早于etime
  - 缓存命中：不执行任何操作
- 如果有重载策略：
  - 全量：从L3-LMDB中获取etime之前的count条数据，追加到L1缓存tickList的末尾
  - 增量：从L3-LMDB中获取 \(L1缓存上次更新时间, etime\] 之间的数据，追加到L1缓存tickList的末尾
- 更新L1缓存tickList的最后更新时间为etime
- 返回L1缓存tickList中的tick切片
```cpp
/**
 * @brief 读取Tick数据切片
 * 
 * 读取指定合约的最新Tick数据，支持智能缓存和增量加载。
 * 该方法是实时数据读取器的核心功能之一。
 * 
 * @param stdCode 标准合约代码（如"SHFE.rb.HOT"）
 * @param count 需要读取的Tick数量
 * @param etime 结束时间（0表示当前时间）
 * @return Tick数据切片对象，失败返回NULL
 */
WTSTickSlice* WtDataReaderAD::readTickSlice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
```

#### 从 L1-K线缓存中读取K线数据 readKlineSlice

流程：
- 获取 L1-K线缓存 `_bars_cache` 中对应 stdCode, count 的数据块引用 barsList
- 如果要求数量 count 大于该L1数据块的容量：
  - 重置该数据块，从 L3-LMDB 中加载到 barsList
- 如果 barList 的上次更新时间小于 etime 并且 barList 中最后一根K线时间小于 etime：
  - 从 L3-LMDB 中加载来增量更新 barsList
  - 如果 barList 中最后一根K线时间仍然小于 etime：获取L2-实时K线缓存中的数据 rtBar: WTSBarStruct*
    - 如果 rtBar 的第一根K线时间大于 etime（这段时间内L2的数据已经转移到L3了）
      - 从 L3-LMDB 中加载来增量更新 barsList
    - 否则
      - 将 rtBar 添加到 barsList
- 更新 barsList 的上次更新时间为 etime
- 返回 barsList 中的K线数据
```cpp
/**
 * @param stdCode 标准合约代码（如"SHFE.rb.HOT"）
 * @param period K线周期（KP_Minute1、KP_Minute5、KP_DAY等）
 * @param count 需要读取的K线数量
 * @param etime 结束时间（0表示当前时间）
 * @return K线数据切片对象，失败返回NULL
 */
WTSKlineSlice* WtDataReaderAD::readKlineSlice(const char* stdCode, WTSKlinePeriod period, uint32_t count, uint64_t etime /* = 0 */)
```

#### L2/L3 更新 L1-K线缓存 onMinuteEnd

流程：
- 如果当前时间大于上次处理时间 `_last_time`
  - 使用 L2/L3 更新 L1-K线缓存 `_bars_cache` 
  - `_sink` 触发回调 on_all_bar_updated
  - 上次处理时间更新为当前时间
```cpp
/**
 * @param uDate 当前日期（YYYYMMDD格式）
 * @param uTime 当前时间（HHmm格式）
 * @param endTDate 结束交易日（可选，用于日线处理）
 */
void WtDataReaderAD::onMinuteEnd(uint32_t uDate, uint32_t uTime, uint32_t endTDate /* = 0 */)
```

### 数据库管理

#### 获取 K 线数据库连接 get_k_db

#### 获取 Tick 数据库连接 get_t_db

### 辅助方法

#### 从 L3-LMDB 中加载 K 线数据到 L1-K 线缓存 cacheBarsFromStorage
从 L3-LMDB 中加载 count 条，对应 stdCode 和 period 的 K 线数据到 L1-K 线缓存 `_bars_cache[key]` 中
```cpp
/* @param key 缓存键值
 * @param stdCode 标准合约代码
 * @param period K线周期
 * @param count 需要加载的K线数量
 * @return 加载成功返回true，失败返回false
 */
bool WtDataReaderAD::cacheBarsFromStorage(const std::string& key, const char* stdCode, WTSKlinePeriod period, uint32_t count)
```

#### 从 L3-LMDB 中加载 K 线数据 update_cache_from_lmdb
从 L3-LMDB 中加载对应 stdCode、code、period 的 K 线数据来增量更新 barsList: BarsList
```cpp
/**
 * @param barsList K线列表引用
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param period K线周期
 * @param lastBarTime 最后K线时间（输入输出参数，会被更新为最新的K线时间）
 */
void WtDataReaderAD::update_cache_from_lmdb(BarsList& barsList, const char* exchg, const char* code, WTSKlinePeriod period, uint32_t& lastBarTime)
```

#### 读取 L3-LMDB 中的 K 线数据 read_bars_to_buffer
从 L3-LMDB 中读取对应 stdCode、code、period 的 K 线数据来构成一个 std::string 返回
```cpp
/**
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param period K线周期
 * @return 包含K线数据的字符串缓冲区，失败返回空字符串
 */
std::string WtDataReaderAD::read_bars_to_buffer(const char* exchg, const char* code, WTSKlinePeriod period)
```

#### 获取L2-实时K线缓存中的数据 get_rt_cache_bar
获取L2-实时K线缓存 `_d1_cache/_m1_cache/_m5_cache`（根据 period）中对应 exchg, code 的数据
```cpp
/**
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param period K线周期
 * @return K线数据指针，失败返回NULL
 */
WTSBarStruct* WtDataReaderAD::get_rt_cache_bar(const char* exchg, const char* code, WTSKlinePeriod period)
```

# 数据写入器 WtDataWriterAD.h/cpp
```cpp
class WtDataWriter : public IDataWriter
```
参考 [Includes/note.ipynb/数据管理接口层/数据写入 IDataWriter.h/数据写入接口类 IDataWriter](../Includes/note.ipynb)

## 成员
- **核心管理器指针**
  - `IBaseDataMgr* _bd_mgr`
    - **作用**：指向基础数据管理器。
    - **意义**：用于在写入数据前，获取合约的交易时段 (`WTSSessionInfo`)、合约类型 (`WTSContractInfo`) 等静态信息
  - `IDataWriterSink* _sink`（继承自 `IDataWriter`）
    - **作用**：指向回调接收器（通常是 `DataManager` 实例）。
    - **意义**：用于回传日志 (`outputLog`)，并在数据写入成功后触发数据广播 (`broadcastTick` 等)。
  - `ExtDumpers _dumpers`（继承自 `IDataWriter`）
    - **作用**：存储额外的历史数据转储器。
    - **意义**：允许主存储（LMDB）之外的扩展存储，例如同时写入 CSV 或 HDF5 (此功能由 `IHisDataDumper` 接口 定义)。

- **L2 实时 Tick 缓存**
  - `StdUniqueMutex _mtx_tick_cache`
    - **作用**：一个互斥锁，专门用于保护 `_tick_cache_block` 和 `_tick_cache_idx`。
    - **意义**：防止在多线程（尽管此模块主要是单线程写入）或扩容时发生数据竞争，确保 Tick 缓存索引和数据的一致性。
  - `std::string _cache_file_tick`
    - **作用**：存储 L2 Tick 缓存的文件名，如 `"cache_tick.dmb"`。
  - `wt_hashmap<std::string, uint32_t> _tick_cache_idx`
    - **作用**：**Tick 缓存的核心索引**。
    - **意义**：它是一个哈希表，键 (Key) 是合约代码 (如 "SHFE.rb2305")，值 (Value) 是该合约在 `_tick_cache_block->_items` 数组中的**索引位置** (如 `3`)。这使得 `getCurTick` 方法可以实现 O(1) 复杂度的最新 Tick 查询。
  - `BoostMFPtr _tick_cache_file`
    - **作用**：指向内存映射文件（`BoostMappingFile`）的智能指针。
    - **意义**：管理 `cache_tick.dmb` 文件的内存映射句柄，确保文件被正确映射和释放。
  - `RTTickCache* _tick_cache_block`
    - **作用**：指向内存映射文件在内存中的**起始地址**，并被转换为 `RTTickCache` 结构体指针。
    - **意义**：这是 `WtDataWriterAD` 实际写入**最新 Tick** 的地方。`WtDataReaderAD` 也会读取此内存块以获取当日的 Tick 数据。

- **L2 实时 K 线缓存**
  - `RTBarCacheWrapper _m1_cache`：实时 1 分钟 K 线缓存。
  - `RTBarCacheWrapper _m5_cache`：实时 5 分钟 K 线缓存。
  - `RTBarCacheWrapper _d1_cache`：实时日 K 线缓存。
    - **作用**：这三个成员用于管理 1 分钟、5 分钟和日线的实时 K 线缓存文件（如 `cache_m1.dmb`）。
    - **意义**：`updateBarCache` 方法会从 Tick 实时合成 K 线，并将**当前尚未闭合的最新 K 线**写入这些缓存中。`WtDataReaderAD` 会读取它们以提供给策略。
    - `typedef struct _RTBarCacheWrapper` 包含：
      - `StdUniqueMutex _mtx`：线程锁，保护此 K 线周期的缓存。
      - `std::string _filename`：缓存文件名（如 "cache_m1.dmb"）。
      - `wt_hashmap<std::string, uint32_t> _idx`：合约到K线数组的索引。
      - `BoostMFPtr _file_ptr`：内存映射文件句柄。
      - `RTBarCache* _cache_block`：指向内存映射地址，对应 `RTBarCache` 结构。

- **异步任务处理**
  - `std::queue<TaskInfo> _tasks`
    - **作用**：任务队列，`TaskInfo` 是 `std::function<void()>`。
    - **意义**：`writeTick` 接收到数据后，将K线合成、数据库写入等耗时操作打包成一个 `TaskInfo` 放入此队列，然后立即返回，**不阻塞行情线程**。
  - `StdThreadPtr _task_thrd`
    - **作用**：后台工作线程。
    - **意义**：该线程是实际的“工作者”，它不断地从 `_tasks` 队列中取出任务并执行。
  - `StdUniqueMutex _task_mtx` & `StdCondVariable _task_cond`
    - **作用**：用于 `_tasks` 队列的线程同步锁和条件变量。
    - **意义**：实现高效的生产者-消费者模型。`writeTick` 是生产者，`_task_thrd` 是消费者。

- **L3 存储**
  - `WtLMDBMap _exchg_m1_dbs`：1 分钟 K 线数据库连接池（按交易所索引）。
  - `WtLMDBMap _exchg_m5_dbs`：5 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_d1_dbs`：日 K 线数据库连接池。
  - `WtLMDBMap _tick_dbs`：Tick 数据库连接池（按合约索引）。
    - typedef wt_hashmap\<std::string, `WtLMDBPtr`\> WtLMDBMap：数据库映射表类型
    - typedef std::shared_ptr\<`WtLMDB`\> WtLMDBPtr：LMDB数据库智能指针类型
  - **作用**：这些成员是 `wt_hashmap`，用于缓存 LMDB 数据库的连接句柄 (`WtLMDBPtr`)。
  - **意义**：`get_k_db` 和 `get_t_db` 方法使用它们实现**数据库连接池**，避免每次写入都重新打开数据库文件，极大提升了写入性能。

- **配置与状态**
  - `std::string _base_dir`：数据存储根目录。
  - `bool _terminated`：线程终止标志。
  - `bool _disable_tick`, `_disable_min1`, `_disable_min5`, `_disable_day`
    - **作用**：功能开关。
    - **意义**：允许用户在配置中禁用特定数据的写入，例如，如果只关心分钟线，可以设置 `"disabletick": true` 来节省磁盘空间。
  - `uint32_t _tick_mapsize`, `_kline_mapsize`
    - **作用**：LMDB 数据库的预分配空间大小。
    - **意义**：影响 LMDB 性能的关键参数。

## 方法

### IDataWriter 接口实现

#### 初始化写入器 init

#### 写入Tick数据 writeTick
添加任务：
- 如果 curTick 对应的合约当前在其交易时段内
  - 使用 curTick 更新 L2-Tick 缓存
  - 将 curTick 存入 L3-LMDB 数据块，并触发 `_dumpers` 中所有扩展转储器的回调 dumpHisTicks
  - 使用 curTick 合成并更新 L2-K 线缓存
  - 触发 `_sink` 的 broadcastTick 广播到下游系统
```cpp
/**
 * @param curTick Tick数据对象
 * @param procFlag 处理标志：
 *                 - 0: 直接使用原始数据
 *                 - 1: 计算增量数据（成交量、成交额等）
 *                 - 2: 自动累加模式
 * @return 处理成功返回true，失败返回false
 */
bool WtDataWriterAD::writeTick(WTSTickData* curTick, uint32_t procFlag)
```

### 异步与后台处理

#### 推送任务 pushTask
- 如果是异步模式（`_async_task` 为 true）
  - 将任务 task 放到 `_tasks`，条件变量 `_task_cond` 通知
- 否则：直接执行 task
- 如果线程 `_task_thrd` 还没创建，创建线程函数：
  - `_tasks` 为空时，等待：直到条件变量 `_task_cond` 通知
  - 被通知后，从 `_tasks` 中取出所有任务执行

```cpp
/**
 * @brief 推送异步任务
 * 
 * 将任务添加到异步处理队列中，由后台线程执行。
 * 如果是第一次调用，会自动创建后台处理线程。
 * 
 * @param task 任务函数对象
 */
void WtDataWriterAD::pushTask(TaskInfo task)
```

### L2 缓存管理

#### 初始化 L2 的Tick和K线缓存 loadCache
如果缓存文件不存在会自动创建，如果存在则会加载现有数据并重建索引。
- `_tick_cache_block`
- `_m1_cache`、`_m5_cache`、`_d1_cache`
```cpp
void WtDataWriterAD::loadCache()
```

#### 调整L2缓存块大小 resizeRTBlock
```cpp
/**
 * @tparam HeaderType 缓存块头部类型（如RTTickCache、RTBarCache）
 * @tparam T 数据项类型（如TickCacheItem、BarCacheItem）
 * @param mfPtr 内存映射文件智能指针引用
 * @param nCount 新的容量大小（数据项数量）
 * @return 调整后的内存地址，失败返回NULL
 */
template<typename HeaderType, typename T>
void* WtDataWriterAD::resizeRTBlock(BoostMFPtr& mfPtr, uint32_t nCount)
```

#### 使用Tick数据更新L2-Tick缓存 updateTickCache
```cpp
/**
 * @param ct 合约信息
 * @param curTick 当前Tick数据
 * @param procFlag 处理标志：
 *                 - 0: 直接使用原始数据
 *                 - 1: 计算增量数据（成交量、成交额等）
 *                 - 2: 自动累加模式
 * @return 更新成功返回true，失败返回false
 */
bool WtDataWriterAD::updateTickCache(WTSContractInfo* ct, WTSTickData* curTick, uint32_t procFlag)
```

#### 使用Tick数据合成并更新L2-K线缓存 updateBarCache
```cpp
/**
 * @param ct 合约信息
 * @param curTick 当前Tick数据
 */
void WtDataWriterAD::updateBarCache(WTSContractInfo* ct, WTSTickData* curTick)
```

### L3 数据库写入

#### 写入Tick到L3-LMDB数据库 pipeToTicks
将 curTick 存入 L3-LMDB 数据块，并触发 `_dumpers` 中所有扩展转储器的回调 dumpHisTicks
```cpp
/**
 * @param ct 合约信息
 * @param curTick 当前Tick数据
 */
void WtDataWriterAD::pipeToTicks(WTSContractInfo* ct, WTSTickData* curTick)
```

#### 写入日K线到L3-LMDB数据库 pipeToDayBars

#### 写入1分钟K线到L3-LMDB数据库 pipeToM1Bars

#### 写入5分钟K线到L3-LMDB数据库 pipeToM5Bars

### L3 数据库连接池

#### 获取 K 线数据库连接 get_k_db

#### 获取 Tick 数据库连接 get_t_db

# 回测数据读取器 WtBtDtReaderAD.h/cpp
```cpp
class WtBtDtReader : public IBtDtReader
```
参考 [Includes/note.ipynb/数据管理接口层/回测数据读取 IBtDtReader.h/回测数据读取接口类 IBtDtReader](../Includes/note.ipynb)

## 成员
**成员：**
- **核心管理器指针 (Core Pointers)**
  - `std::string _base_dir`
    - **作用**：数据存储根目录路径。
    - **意义**：用于定位 LMDB 数据库的物理位置。
  - `IBtDtReaderSink* _sink`（继承自 `IBtDtReader`）
    - **作用**：指向回调接收器。
    - **意义**：主要用于回传日志 (`pipe_btreader_log`)。

- **L3 存储 (LMDB 数据库句柄)**
  - `WtLMDBMap _exchg_m1_dbs`：1 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_m5_dbs`：5 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_d1_dbs`：日 K 线数据库连接池。
  - `WtLMDBMap _tick_dbs`：Tick 数据库连接池。
    - typedef wt_hashmap\<std::string, `WtLMDBPtr`\> WtLMDBMap：数据库映射表类型
    - typedef std::shared_ptr\<`WtLMDB`\> WtLMDBPtr：LMDB数据库智能指针类型
  - **作用**：缓存 LMDB 数据库的连接句柄。
  - **意义**：与 `WtDataWriterAD` 相同，`get_k_db` 和 `get_t_db` 方法利用此连接池按需打开数据库（以**只读模式**）。
  - **区别**：`WtBtDtReaderAD` 的实现非常纯粹，**它没有任何 L1 或 L2 缓存**。它的所有 `read_raw_...` 方法都是直接查询 LMDB 数据库，将原始二进制数据块读入 `std::string& buffer` 中并立即返回。

## 方法

### IBtDtReader 接口实现

#### 初始化读取器 init

#### 读取原始 K 线数据 read_raw_bars

#### 读取原始 Tick 数据 read_raw_ticks

### 私有数据库管理

#### 获取 K 线数据库连接 get_k_db

#### 获取 Tick 数据库连接 get_t_db

# 随机数据读取器 WtRdmDtReaderAD.h/cpp
```cpp
class WtRdmDtReader : public IRdmDtReader
```
参考 [Includes/note.ipynb/数据管理接口层/随机顺序数据读取 IRdmDtReader.h/随机顺序数据读取接口类 IRdmDtReader](../Includes/note.ipynb)

## 成员
**成员：**
- **核心管理器指针 (Core Pointers)**
  - `std::string _base_dir`：数据存储根目录路径。
  - `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针。
  - `IHotMgr* _hot_mgr`：主力合约管理器指针。
  - `IRdmDtReaderSink* _sink`（继承自 `IRdmDtReader`）
    - **作用**：指向回调接收器。
    - **意义**：用于回传日志，并为 `_base_data_mgr` 和 `_hot_mgr` 提供访问入口。

- **L1 内存缓存 (In-Memory Cache)**
  - `BarsCache _bars_cache`
    - **作用**：K 线数据的**内存缓存 (L1)**。键为 "合约代码#周期"。
    - **意义**：`readKlineSliceByRange` 会将从 LMDB 中查到的数据加载到这里。
    - `typedef wt_hashmap<std::string, BarsList> BarsCache;`
      - `struct BarsList` 包含：
        - `uint64_t _last_bar_time`：缓存中最后一条 K 线的时间戳。
        - `std::vector<WTSBarStruct> _bars`：**核心数据区**。使用 `std::vector` 存储 K 线，**支持动态增量扩展**。
  - `TicksCache _ticks_cache`
    - **作用**：Tick 数据的**内存缓存 (L1)**。键为 "交易所.合约"。
    - **意义**：`readTickSliceByRange` 会将 LMDB 数据加载到这里。
    - `typedef wt_hashmap<std::string, TicksList> TicksCache;`
      - `struct TicksList` 包含：
        - `uint64_t _first_tick_time`：缓存中第一条 Tick 的时间戳。
        - `uint64_t _last_tick_time`：缓存中最后一条 Tick 的时间戳。
        - `std::vector<WTSTickStruct> _ticks`：**核心数据区**。
    - **智能缓存逻辑**：
      当用户请求一个时间范围时，`WtRdmDtReaderAD` 会检查 `_first_tick_time` 和 `_last_tick_time`。
      - 如果请求范围完全在缓存内，直接从 `_ticks` vector 返回切片。
      - 如果请求范围超出了缓存（`bNeedOlder` 或 `bNeedNewer`），它**只会从 LMDB 加载缺失的部分**，并将其拼接到 `_ticks` vector 的头部或尾部，从而实现智能增量加载。

- **L3 存储 (LMDB 数据库句柄)**
  - `WtLMDBMap _exchg_m1_dbs`：1 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_m5_dbs`：5 分钟 K 线数据库连接池。
  - `WtLMDBMap _exchg_d1_dbs`：日 K 线数据库连接池。
  - `WtLMDBMap _tick_dbs`：Tick 数据库连接池。
    - typedef wt_hashmap\<std::string, `WtLMDBPtr`\> WtLMDBMap：数据库映射表类型
    - typedef std::shared_ptr\<`WtLMDB`\> WtLMDBPtr：LMDB数据库智能指针类型
  - **作用**：与 `WtBtDtReaderAD` 相同，用于缓存 LMDB 的**只读**连接句柄。
  - **意义**：作为 L1 缓存的数据来源 (L3)。

## 方法

### IRdmDtReader 接口实现

#### 初始化读取器 init

#### 按时间范围读取 Tick 数据 readTickSliceByRange

#### 按时间范围读取 K 线数据 readKlineSliceByRange

#### 按数量读取 Tick 数据 readTickSliceByCount

#### 按数量读取 K 线数据 readKlineSliceByCount

#### 按日期读取 Tick 数据 readTickSliceByDate

### 私有数据库管理

#### 获取 K 线数据库连接 get_k_db

#### 获取 Tick 数据库连接 get_t_db